# Cloud Logging  

Google's logging service is a central service to store and retrieve information. Logged events can help syncronize state and actions as well as provide status and debugging information. In Python it can be used either directly via the Google module or with the builtin `logging` module. Refer to the `logging` module documentation for usage patterns apply within Python; the concepts described apply equally to Google Cloud Logging. 

## Namespaces  
The `logging` module is part of the standard library in Python. In this example, it will be imported and referenced as `logging`.

The Google Cloud Platform Python module for Cloud Logging is `import google.cloud.logging` which can present a name shadowing issue. In this example Cloud Logging will be aliased as `gcp_logging`

Finally, the package `pygcp` has a Cloud Logging module. To avoid confusion it is named `cloud_logging` and can be imported with that name.  

## Logging Attributes  
When logging events to GCP, there are several attributes of a log entry. Most details below are from the API documents, available at: https://cloud.google.com/logging/docs/reference/v2/rest/v2/LogEntry  
The following data relates to the information returned by `pygcp.cloud_logging.query`.

### Log Name  
The name of the log. The format will be one of the following depending on the source of the event:  
```
"projects/[PROJECT_ID]/logs/[LOG_ID]"
"organizations/[ORGANIZATION_ID]/logs/[LOG_ID]"
"billingAccounts/[BILLING_ACCOUNT_ID]/logs/[LOG_ID]"
"folders/[FOLDER_ID]/logs/[LOG_ID]"
```
From the API docs:  
A project number may be used in place of PROJECT_ID. The project number is translated to its corresponding PROJECT_ID internally and the logName field will contain PROJECT_ID in queries and exports.

[LOG_ID] must be URL-encoded within logName. Example: "organizations/1234567890/logs/cloudresourcemanager.googleapis.com%2Factivity".

[LOG_ID] must be less than 512 characters long and can only include the following characters: upper and lower case alphanumeric characters, forward-slash, underscore, hyphen, and period.

### Timestamp  
From the API Docs:  
The time the event described by the log entry occurred. This time is used to compute the log entry's age and to enforce the logs retention period. If this field is omitted in a new log entry, then Logging assigns it the current time. Timestamps have nanosecond accuracy, but trailing zeros in the fractional seconds might be omitted when the timestamp is displayed.

Incoming log entries must have timestamps that don't exceed the logs retention period in the past, and that don't exceed 24 hours in the future. Log entries outside those time boundaries are rejected by Logging.

A timestamp in RFC3339 UTC "Zulu" format, with nanosecond resolution and up to nine fractional digits. Examples: "2014-10-02T15:01:23Z" and "2014-10-02T15:01:23.045123456Z".

### Severity  
Google Cloud Platform uses a standardized set of levels to allow filter for a specific kind of event. They are:  

#### Decending Levels of Cloud Logging Severity  

    Emergency  
    Alert  
    Critical  
    Error  
    Warning  
    Notice  
    Info  
    Debug  
    Default  

### Labels  
User specified `key:value` labels for the event. More than one label can be applied to an event. The labels can be user-defined or system-defined.

User-defined labels are arbitrary key, value pairs that you can use to classify logs.

System-defined labels are defined by GCP services for platform logs. They have two components - a service namespace component and the attribute name. For example: compute.googleapis.com/resourceName.

Cloud Logging truncates label keys that exceed 512 B and label values that exceed 64 KB upon their associated log entry being written. The truncation is indicated by an ellipsis at the end of the character string.

### Logging Query Language  
String values must be enclosed in double quotes, not single quotes.  
"yes" is correct  
'no' is not a valid string  
Examples below use brackets to show variables inserted into the query syntax.  

a = this  
b = that  
query = {a}+{b} = this+that

#### Labels
`key` : str, key name of label `key:value` pair  
`value` : str, value of label `key:value` pair  
Query format: `labels{key}="{value}"`  

### Payload  
The specific message, data or structured data associated with the event. If creating events, can be anything but probably best to use JSON strings.  

In [1]:
import pandas as pd

import google.cloud.logging as gcp_logging

from pygcp import credentials
from pygcp import cloud_logging
from pygcp import bq
from pygcp import utils

In [2]:
pd.options.display.max_columns = None
pd.options.display.max_rows = None
pd.options.display.max_colwidth = 1000

In [3]:
%store -r cred_file
%store -r project_id

In [4]:
gcp_credentials, project_id = credentials.create_credentials(cred_file=cred_file, 
                                                             project_id=project_id)

In [5]:
gcp_credentials, gcp_credentials.project_id, project_id

(<google.oauth2.service_account.Credentials at 0x7f03efc06410>,
 'pygcp-407281',
 'pygcp-407281')

### 1. Google Cloud Logging Module  

From the API docs: 

```
google.cloud.logging_v2.client.Client  

Parameters
----------
project (Optional[str]) - the project which the client acts on behalf of. If not passed, falls back to the default inferred from the environment.

credentials (Optional[google.auth.credentials.Credentials]) - The OAuth2 Credentials to use for this client. If not passed (and if no _http object is passed), falls back to the default inferred from the environment.
```



In [6]:
gcp_logging_client = gcp_logging.Client(credentials=gcp_credentials,
                                                 project=project_id)

In [7]:
gcp_logging_client.project

'pygcp-407281'

In [8]:
gcp_logger = gcp_logging_client.logger(name="gcp")

In [9]:
gcp_logger.name

'gcp'

In [10]:
gcp_logger.labels = {'gcp_label_A': 'label_value_1'}

In [11]:
gcp_logger.labels

{'gcp_label_A': 'label_value_1'}

In [12]:
gcp_logger.log(f'Google Cloud Logging module - Info Level',
               severity='Info'
               )

In [13]:
gcp_logger.log(f'Google Cloud Logging module - Warning Level',
               severity='Warning'
               )

In [14]:
gcp_logger.log(f'Google Cloud Logging module - Error Level',
               severity='Error'
               )

In [15]:
gcp_logger.labels = {'gcp_label_A': 'value_1',
                     'gcp_label_B': 'value_2'}

In [16]:
gcp_logger.log(f'Google Cloud Logging module - Info Level with 2 labels',
               severity='Info'
               )

In [17]:
gcp_logger.labels = {'gcp_label_C': 'value_3',
                     'gcp_label_D': 'value_4'}

In [18]:
gcp_logger.log(f'Google Cloud Logging module - Warning Level with 2 labels',
               severity='Warning'
               )

### 3. Python Logging Library  
By attaching the Cloud Logging session that we created and authenticated into earlier, we can use Python's logging module methods. It is important that `.setup_logging()` is called *before* importing the Python `logging` module. By default, it will collect all information at the `INFO` level and higher.

In [19]:
gcp_logging_client.setup_logging()

In [20]:
import logging

In [21]:
pylogger = logging.getLogger('python')

In [22]:
pylogger.info(f'Python Logging module - Info Level')

In [23]:
pylogger.warning(f'Python Logging module - Warning Level')

In [24]:
pylogger.info(f'Python Logging module - Info Level',
              extra={"labels":{'python_label_A': 'value_5',
                               'python_label_B': 'value_6'}
                    }
    )

In [25]:
pylogger.warning(f'Python Logging module - Warning Level',
                 extra={"labels":{'python_label_C': 'value_7',
                                  'python_label_D': 'value_8'}
                       }
    )

### Example: Writing Metadata to Cloud Logging  
Google's BigQuery data warehouse keeps a record of it's own events in the INFORMATION_SCHEMA tables so this example isn't the best way of querying BigQuery job information. However for this logging example it's very complete metadata and a good example of the detail that can be inserted into Cloud Logging events.

In [26]:
query_text = """
SELECT COUNT(*) as row_count 
FROM `bigquery-public-data.noaa_preliminary_severe_storms.wind_reports`
"""

In [27]:
metadata_query, _df = bq.to_df(gcp_credentials=gcp_credentials,
                               project_id=project_id,
                               query_text=query_text)

BigQuery query bytes: 0it [00:00, ?it/s]


The complete metadata payload returned by BigQuery is:

And to log this metadata with the `info` level we use the same pattern as earlier. One difference between the Cloud Logging module and the built-in logging library is type handling. The built-in module accepts strings, whereas the Cloud Logging module will accept JSON and dict data as well.

In [28]:
type(metadata_query)

dict

In [29]:
pylogger.info(metadata_query, extra={"labels":{'python_label_E': 'BigQuery Metadata'}})

### Read Cloud Logging Records  
Note: if this fails, you might need to request a higher quota.  

Here we use the Cloud Logging API to retrieve a list of logging events converted to a Pandas DataFrame. We request a few different event levels and then further filter the results in Pandas. Note that the `labels` column is a JSON string and the `payload` columns is a dictionary.

In [30]:
df = cloud_logging.query(gcp_credentials=gcp_credentials,
                            project_id=project_id,
                            severity=['warning', 'info', 'debug'])

In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 167 entries, 0 to 166
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype              
---  ------     --------------  -----              
 0   log_name   167 non-null    object             
 1   timestamp  167 non-null    datetime64[ns, UTC]
 2   severity   167 non-null    object             
 3   labels     47 non-null     object             
 4   payload    167 non-null    object             
dtypes: datetime64[ns, UTC](1), object(4)
memory usage: 6.6+ KB


In [32]:
df.head()

,log_name,timestamp,severity,labels,payload
0,projects/pygcp-407281/logs/gcp,2023-12-12 03:15:38.103891+00:00,INFO,{'gcp_label_A': 'label_value_1'},Google Cloud Logging module - Info Level
1,projects/pygcp-407281/logs/gcp,2023-12-12 03:15:38.227200+00:00,WARNING,{'gcp_label_A': 'label_value_1'},Google Cloud Logging module - Warning Level
2,projects/pygcp-407281/logs/gcp,2023-12-12 03:15:38.500694+00:00,INFO,"{'gcp_label_B': 'value_2', 'gcp_label_A': 'value_1'}",Google Cloud Logging module - Info Level with 2 labels
3,projects/pygcp-407281/logs/python,2023-12-12 03:15:38.524989+00:00,INFO,{'python_logger': 'python'},Python Logging module - Info Level
4,projects/pygcp-407281/logs/python,2023-12-12 03:15:38.532085+00:00,WARNING,{'python_logger': 'python'},Python Logging module - Warning Level


In [38]:
df.loc[df.labels.apply(lambda r: utils.has_keys(r, 'gcp_label_A'))].head()

,log_name,timestamp,severity,labels,payload
0,projects/pygcp-407281/logs/gcp,2023-12-12 03:15:38.103891+00:00,INFO,{'gcp_label_A': 'label_value_1'},Google Cloud Logging module - Info Level
1,projects/pygcp-407281/logs/gcp,2023-12-12 03:15:38.227200+00:00,WARNING,{'gcp_label_A': 'label_value_1'},Google Cloud Logging module - Warning Level
2,projects/pygcp-407281/logs/gcp,2023-12-12 03:15:38.500694+00:00,INFO,"{'gcp_label_B': 'value_2', 'gcp_label_A': 'value_1'}",Google Cloud Logging module - Info Level with 2 labels
16,projects/pygcp-407281/logs/gcp,2023-12-12 03:45:39.423760+00:00,INFO,{'gcp_label_A': 'label_value_1'},Google Cloud Logging module - Info Level
17,projects/pygcp-407281/logs/gcp,2023-12-12 03:45:39.557199+00:00,WARNING,{'gcp_label_A': 'label_value_1'},Google Cloud Logging module - Warning Level


In [39]:
df.loc[df.labels.apply(lambda r: utils.has_values(r, 'value_1'))].head()

,log_name,timestamp,severity,labels,payload
2,projects/pygcp-407281/logs/gcp,2023-12-12 03:15:38.500694+00:00,INFO,"{'gcp_label_B': 'value_2', 'gcp_label_A': 'value_1'}",Google Cloud Logging module - Info Level with 2 labels
18,projects/pygcp-407281/logs/gcp,2023-12-12 03:45:39.834121+00:00,INFO,"{'gcp_label_B': 'value_2', 'gcp_label_A': 'value_1'}",Google Cloud Logging module - Info Level with 2 labels
60,projects/pygcp-407281/logs/gcp,2023-12-12 18:49:26.566181+00:00,INFO,"{'gcp_label_B': 'value_2', 'gcp_label_A': 'value_1'}",Google Cloud Logging module - Info Level with 2 labels
73,projects/pygcp-407281/logs/gcp,2023-12-12 19:43:19.640465+00:00,INFO,"{'gcp_label_B': 'value_2', 'gcp_label_A': 'value_1'}",Google Cloud Logging module - Info Level with 2 labels
134,projects/pygcp-407281/logs/gcp,2023-12-12 20:04:02.501634+00:00,INFO,"{'gcp_label_B': 'value_2', 'gcp_label_A': 'value_1'}",Google Cloud Logging module - Info Level with 2 labels


In [35]:
selection_dict = {'python_label_D': 'value_8', 'python_logger': 'python', 'python_label_C': 'value_7'}
df.loc[df.labels.apply(lambda r: utils.has_keys_and_values(r, selction_dict))]

,log_name,timestamp,severity,labels,payload
7,projects/pygcp-407281/logs/python,2023-12-12 03:16:42.197309+00:00,WARNING,"{'python_label_D': 'value_8', 'python_logger': 'python', 'python_label_C': 'value_7'}",Python Logging module - Info Level
23,projects/pygcp-407281/logs/python,2023-12-12 03:45:40.512859+00:00,WARNING,"{'python_label_D': 'value_8', 'python_logger': 'python', 'python_label_C': 'value_7'}",Python Logging module - Info Level
65,projects/pygcp-407281/logs/python,2023-12-12 18:49:32.161501+00:00,WARNING,"{'python_label_D': 'value_8', 'python_logger': 'python', 'python_label_C': 'value_7'}",Python Logging module - Warning Level
71,projects/pygcp-407281/logs/python,2023-12-12 19:43:19.172669+00:00,WARNING,"{'python_label_D': 'value_8', 'python_logger': 'python', 'python_label_C': 'value_7'}",Python Logging module - Warning Level
131,projects/pygcp-407281/logs/python,2023-12-12 20:04:01.720067+00:00,WARNING,"{'python_label_D': 'value_8', 'python_logger': 'python', 'python_label_C': 'value_7'}",Python Logging module - Warning Level


## BigQuery Sink for Cloud Logging  
Recall we created two loggers:  
`gcp_logger = gcp_logging_client.logger(name='gcp')`  
`pylogger = logging.getLogger('python')`  

In this demo, we used `project_id = pygcp-407281`; within this project we created a BigQuery dataset called `cl` and using the cloud console set it as a Cloud Logging BigQuery sink. When using the Cloud Logging to BigQuery sink, tables are created for each named logger. Therefore, the table names for our two loggers are:  
`gcp_logger = gcp_logging_client.logger(name='gcp') = pygcp-407281.cl.gcp`  
`pylogger = logging.getLogger('python') = pygcp-407281.cl.python`  

When creating the sink, we chose time partitioning. This will place all data from a specific logger name in one table. If not selected during sink creation, a new table will be created every day with a timestamped suffix (`name-YYYYMMDD`) which allows wildcard queries on the table names. Both approaches limit total rows queried, assuming proper time range selections are included in the query SQL.

Looking at all the tables created by Cloud Logging:

In [36]:
bq.inventory(gcp_credentials, project_id)[['project_id', 'dataset_id', 'table_id']]

BigQuery query bytes: 0it [00:00, ?it/s]


,project_id,dataset_id,table_id
0,pygcp-407281,cl,cloudaudit_googleapis_com_activity
1,pygcp-407281,cl,cloudaudit_googleapis_com_data_access
2,pygcp-407281,cl,gcp
3,pygcp-407281,cl,python


We'll query the two tables `gcp` and `python`, the others are created by GCP services.

In [40]:
query_text = """SELECT * FROM `pygcp-407281.cl.gcp`"""
metadata_query, df_download = bq.to_df(gcp_credentials=gcp_credentials,
                                       project_id=project_id,
                                       query_text=query_text)
df_download.head()

BigQuery query bytes: 100%|██████████| 2570/2570 [00:00<00:00, 2004343.86it/s]


,logName,resource,textPayload,timestamp,receiveTimestamp,severity,insertId,httpRequest,labels,operation,trace,spanId,traceSampled,sourceLocation,split,errorGroups,jsonPayload
0,projects/pygcp-407281/logs/gcp,"{'type': 'global', 'labels': {'project_id': 'pygcp-407281'}}",Google Cloud Logging module - Warning Level,2023-12-12 03:45:39.557199+00:00,2023-12-12 03:45:39.557199+00:00,WARNING,19iru7mfhq3omy,None,"{'gcp_label_a': 'label_value_1', 'gcp_label_d': None, 'gcp_label_c': None, 'gcp_label_b': None}",None,None,None,<NA>,None,None,[],None
1,projects/pygcp-407281/logs/gcp,"{'type': 'global', 'labels': {'project_id': 'pygcp-407281'}}",Google Cloud Logging module - Error Level,2023-12-12 03:45:39.689558+00:00,2023-12-12 03:45:39.689558+00:00,ERROR,1t7gbtmfkadzig,None,"{'gcp_label_a': 'label_value_1', 'gcp_label_d': None, 'gcp_label_c': None, 'gcp_label_b': None}",None,None,None,<NA>,None,None,[],None
2,projects/pygcp-407281/logs/gcp,"{'type': 'global', 'labels': {'project_id': 'pygcp-407281'}}",Google Cloud Logging module - Warning Level with 2 labels,2023-12-12 03:45:39.951781+00:00,2023-12-12 03:45:39.951781+00:00,WARNING,1tru5u4fgvg5ep,None,"{'gcp_label_a': None, 'gcp_label_d': 'value_4', 'gcp_label_c': 'value_3', 'gcp_label_b': None}",None,None,None,<NA>,None,None,[],None
3,projects/pygcp-407281/logs/gcp,"{'type': 'global', 'labels': {'project_id': 'pygcp-407281'}}",Google Cloud Logging module - Info Level,2023-12-12 03:45:39.423760+00:00,2023-12-12 03:45:39.423760+00:00,INFO,tett7tfkcf5eh,None,"{'gcp_label_a': 'label_value_1', 'gcp_label_d': None, 'gcp_label_c': None, 'gcp_label_b': None}",None,None,None,<NA>,None,None,[],None
4,projects/pygcp-407281/logs/gcp,"{'type': 'global', 'labels': {'project_id': 'pygcp-407281'}}",Google Cloud Logging module - Info Level with 2 labels,2023-12-12 03:45:39.834121+00:00,2023-12-12 03:45:39.834121+00:00,INFO,g3e8xuf2u6965,None,"{'gcp_label_a': 'value_1', 'gcp_label_d': None, 'gcp_label_c': None, 'gcp_label_b': 'value_2'}",None,None,None,<NA>,None,None,[],None


Note that the labels are populated into the `labels` column as a sparse `STRUCT` type which becomes a `dict` type in Pandas. 

In [41]:
query_text = """SELECT * FROM `pygcp-407281.cl.python`"""
metadata_query, df_download = bq.to_df(gcp_credentials=gcp_credentials,
                                       project_id=project_id,
                                       query_text=query_text)
df_download.head()

BigQuery query bytes: 100%|██████████| 4406/4406 [00:00<00:00, 3751543.53it/s]


,logName,resource,textPayload,timestamp,receiveTimestamp,severity,insertId,httpRequest,labels,operation,trace,spanId,traceSampled,sourceLocation,split,errorGroups,jsonPayload
0,projects/pygcp-407281/logs/python,"{'type': 'global', 'labels': {'project_id': 'pygcp-407281'}}",Python Logging module - Info Level,2023-12-12 18:49:31.068255+00:00,2023-12-12 18:49:31.841409+00:00,INFO,11qsenmfho9v0x,None,"{'python_logger': 'python', 'python_label_a': 'value_5', 'python_label_b': 'value_6', 'python_label_c': None, 'python_label_d': None, 'python_label_e': None}",None,None,None,<NA>,"{'file': '/tmp/ipykernel_20020/558421562.py', 'line': 1, 'function': '<module>'}",None,[],None
1,projects/pygcp-407281/logs/python,"{'type': 'global', 'labels': {'project_id': 'pygcp-407281'}}",Python Logging module - Warning Level,2023-12-12 18:49:29.343175+00:00,2023-12-12 18:49:30.087452+00:00,WARNING,5a77lkfepa3ai,None,"{'python_logger': 'python', 'python_label_a': None, 'python_label_b': None, 'python_label_c': None, 'python_label_d': None, 'python_label_e': None}",None,None,None,<NA>,"{'file': '/tmp/ipykernel_20020/3669000032.py', 'line': 1, 'function': '<module>'}",None,[],None
2,projects/pygcp-407281/logs/python,"{'type': 'global', 'labels': {'project_id': 'pygcp-407281'}}",Python Logging module - Info Level,2023-12-12 18:49:29.173393+00:00,2023-12-12 18:49:29.920550+00:00,INFO,10cqam6f2mif5a,None,"{'python_logger': 'python', 'python_label_a': None, 'python_label_b': None, 'python_label_c': None, 'python_label_d': None, 'python_label_e': None}",None,None,None,<NA>,"{'file': '/tmp/ipykernel_20020/4281654247.py', 'line': 1, 'function': '<module>'}",None,[],None
3,projects/pygcp-407281/logs/python,"{'type': 'global', 'labels': {'project_id': 'pygcp-407281'}}",Python Logging module - Warning Level,2023-12-12 18:49:32.161501+00:00,2023-12-12 18:49:32.925686+00:00,WARNING,1js1af2fk08w05,None,"{'python_logger': 'python', 'python_label_a': None, 'python_label_b': None, 'python_label_c': 'value_7', 'python_label_d': 'value_8', 'python_label_e': None}",None,None,None,<NA>,"{'file': '/tmp/ipykernel_20020/903531098.py', 'line': 1, 'function': '<module>'}",None,[],None
4,projects/pygcp-407281/logs/python,"{'type': 'global', 'labels': {'project_id': 'pygcp-407281'}}",Python Logging module - Info Level,2023-12-12 19:43:19.150583+00:00,2023-12-12 19:43:20.356263+00:00,INFO,3mgyp1f2skosh,None,"{'python_logger': 'python', 'python_label_a': None, 'python_label_b': None, 'python_label_c': None, 'python_label_d': None, 'python_label_e': None}",None,None,None,<NA>,"{'file': '/tmp/ipykernel_22029/4281654247.py', 'line': 1, 'function': '<module>'}",None,[],None


## Summary  
Using Google Cloud Logging, it is possible to record data and events in Python into a cloud destination. Two methods of recording events were discussed as well as accessing past event data using the Python API and a BigQuery data warehouse.  

### References  
Python Logging module  
https://docs.python.org/3/library/logging.html  

Google Cloud Logging  
https://cloud.google.com/logging/docs/reference/libraries  
https://cloud.google.com/logging/docs/reference/v2/rest/v2/LogEntry  
https://cloud.google.com/logging/docs/view/logging-query-language
https://cloud.google.com/python/docs/reference/logging/latest  
https://cloud.google.com/logging/docs/setup/python  
https://googleapis.dev/python/logging/latest/index.html  
https://cloud.google.com/logging/docs/view/building-queries  
https://github.com/googleapis/python-logging  

PyGCP Package  
https://github.com/crdietrich/pygcp  